# Final Project: Personal Productivity Assistant
Proyek ini membangun asisten chatbot Streamlit berbasis Google Gemini API yang siap dieksekusi di Google Colab.
Chatbot ini memanfaatkan **AI Agents (Function Calling)** untuk mengatur jadwal, mencatat keuangan, menulis catatan rapat, dan mengekstrak informasi dokumen melalui **RAG (Retrieval-Augmented Generation)**.

In [ ]:
!pip install -q streamlit google-generativeai langchain langchain-google-genai langchain-community faiss-cpu pypdf pyngrok tiktoken

In [ ]:
%%writefile app.py
import streamlit as st
import google.generativeai as genai
import os
import tempfile
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

st.set_page_config(page_title="AI Productivity Assistant", page_icon="🤖", layout="wide")

# --- INITIALIZATION ---
if "messages" not in st.session_state: st.session_state.messages = []
if "notes" not in st.session_state: st.session_state.notes = []
if "finance" not in st.session_state: st.session_state.finance = []
if "schedule" not in st.session_state: st.session_state.schedule = []
if "vector_store" not in st.session_state: st.session_state.vector_store = None
if "chat" not in st.session_state: st.session_state.chat = None
if "api_key_valid" not in st.session_state: st.session_state.api_key_valid = False

# --- AGENT TOOLS ---
def add_note(topic: str, content: str) -> str:
    """Simpan catatan kuliah, rapat, atau informasi penting lainnya."""
    st.session_state.notes.append({"topic": topic, "content": content})
    return f"Catatan tentang '{topic}' berhasil disimpan."

def get_notes() -> str:
    """Ambil semua catatan yang disimpan."""
    if not st.session_state.notes: return "Belum ada catatan yang disimpan."
    return "Catatan Anda:\n" + "\n".join([f"- [{n['topic']}] {n['content']}" for n in st.session_state.notes])

def add_finance(tipe: str, amount: int, description: str) -> str:
    """Catat keuangan pribadi. tipe HARUS diisi dengan 'pemasukan' atau 'pengeluaran'."""
    if tipe.lower() not in ['pemasukan', 'pengeluaran']: return "Gagal: Tipe harus 'pemasukan' atau 'pengeluaran'."
    st.session_state.finance.append({"type": tipe.lower(), "amount": amount, "description": description})
    return f"{tipe.capitalize()} sebesar {amount} berhasil dicatat."

def get_finance_summary() -> str:
    """Dapatkan ringkasan laporan keuangan."""
    if not st.session_state.finance: return "Belum ada catatan keuangan."
    pem = sum([i['amount'] for i in st.session_state.finance if i['type'] == 'pemasukan'])
    peng = sum([i['amount'] for i in st.session_state.finance if i['type'] == 'pengeluaran'])
    return f"Total Pemasukan: {pem}\nTotal Pengeluaran: {peng}\nSaldo Saat Ini: {pem - peng}"

def add_schedule(task: str, date_time: str) -> str:
    """Tambahkan jadwal atau pengingat baru."""
    st.session_state.schedule.append({"task": task, "time": date_time})
    return f"Jadwal '{task}' pada '{date_time}' berhasil ditambahkan."

def get_schedule() -> str:
    """Lihat semua jadwal dan pengingat."""
    if not st.session_state.schedule: return "Jadwal kosong."
    return "Jadwal:\n" + "\n".join([f"- {s['task']} ({s['time']})" for s in st.session_state.schedule])

def search_document(query: str) -> str:
    """Gunakan ini HANYA jika Anda ditanya tentang isi dokumen/PDF yang diunggah pengguna."""
    if st.session_state.vector_store is None: return "Belum ada dokumen PDF yang diunggah."
    docs = st.session_state.vector_store.similarity_search(query, k=3)
    context = "\n\n".join([d.page_content for d in docs])
    return f"Konteks relevan:\n{context}"

tools_list = [add_note, get_notes, add_finance, get_finance_summary, add_schedule, get_schedule, search_document]

# --- SIDEBAR & CONFIGURATION ---
with st.sidebar:
    st.header("⚙️ Konfigurasi")
    api_key = st.text_input("Gemini API Key", type="password")
    tone = st.selectbox("Gaya Bahasa (Tone)", ["Profesional & Formal", "Santai & Ramah"])
    st.divider()
    
    st.header("📄 Unggah Dokumen (RAG)")
    uploaded_file = st.file_uploader("Upload file PDF", type=["pdf"])
    if st.button("Proses Dokumen") and api_key and uploaded_file:
        with st.spinner("Memproses dengan RAG..."):
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
                tmp.write(uploaded_file.getvalue())
                tmp_path = tmp.name
            loader = PyPDFLoader(tmp_path)
            chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(loader.load_and_split())
            genai.configure(api_key=api_key)
            embedding_model_name = "models/text-embedding-004"
            for m in genai.list_models():
                if "embedContent" in m.supported_generation_methods:
                    embedding_model_name = m.name
                    break
            embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model_name, google_api_key=api_key)
            st.session_state.vector_store = FAISS.from_documents(chunks, embeddings)
            st.success("Dokumen berhasil diindeks!")
            os.unlink(tmp_path)

# --- MAIN UI ---
st.title("🚀 Personal Productivity Assistant")
st.markdown("Mengelola Jadwal, Catatan, Keuangan, dan Dokumen.")

if api_key:
    genai.configure(api_key=api_key)
    sys_instruct = f"Anda adalah Personal Assistant. Gaya Bahasa: {tone}. Gunakan alat yang tersedia untuk mencatat/memanggil data atau mencari di PDF."
    try:
        model = genai.GenerativeModel(model_name="gemini-2.5-flash", tools=tools_list, system_instruction=sys_instruct)
        if st.session_state.chat is None or st.session_state.api_key_valid == False:
            st.session_state.chat = model.start_chat(enable_automatic_function_calling=True)
            st.session_state.api_key_valid = True
    except Exception as e:
        st.error(f"Error: {e}")
        st.session_state.api_key_valid = False
else:
    st.info("Masukkan API Key Anda di sidebar.")

st.divider()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]): st.markdown(msg["content"])

if prompt := st.chat_input("Apa yang bisa saya bantu?"):
    if not st.session_state.api_key_valid:
        st.error("API Key belum diisi!")
    else:
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"): st.markdown(prompt)
        
        with st.chat_message("assistant"):
            with st.spinner("Berpikir dan mengeksekusi..."):
                try:
                    response = st.session_state.chat.send_message(prompt)
                    st.markdown(response.text)
                    st.session_state.messages.append({"role": "assistant", "content": response.text})
                except Exception as e:
                    st.error(f"Error: {e}")


In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Masukkan authtoken ngrok Anda di sini jika diperlukan
# ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN") 

# Membuka tunnel public ke port default Streamlit (8501)
public_url = ngrok.connect(8501)
print(f"🔗 Buka Chatbot Streamlit Anda di tautan berikut: {public_url}")

# Menjalankan Streamlit secara background
!streamlit run app.py &>/dev/null&